# Notebook 06 — Correlation & Outlier Analysis
**Feature 1.7 — EDA & Data Understanding | HitRadar Pro**

## Mục tiêu
- Tính correlation giữa `target_popularity` (label) và các numeric features.
- Phân tích outliers: duration short/long, loudness dương, tempo/time_signature NULL.
- Đưa ra đề xuất cụ thể cho Feature 1.8 / EPIC 2.

⚠️ **`target_popularity` là LABEL ML — KHÔNG dùng làm input feature.**

In [ ]:
import os, psycopg2, pandas as pd, matplotlib.pyplot as plt
import matplotlib.ticker as mticker

conn = psycopg2.connect(
    host='localhost', port=5432, user='postgres',
    password=os.environ.get('PGPASSWORD', ''), dbname='hitradar'
)
print('Kết nối thành công.')

## 1. Correlation với target_popularity

Dùng `CORR()` aggregate của PostgreSQL để tính trên toàn bộ dataset — tránh phải load 586K rows vào RAM.

In [ ]:
df_corr = pd.read_sql("""
    SELECT
        ROUND(CORR(target_popularity, duration_min)::numeric, 4)      AS duration_min,
        ROUND(CORR(target_popularity, release_year)::numeric, 4)      AS release_year,
        ROUND(CORR(target_popularity, danceability)::numeric, 4)      AS danceability,
        ROUND(CORR(target_popularity, energy)::numeric, 4)            AS energy,
        ROUND(CORR(target_popularity, loudness)::numeric, 4)          AS loudness,
        ROUND(CORR(target_popularity, speechiness)::numeric, 4)       AS speechiness,
        ROUND(CORR(target_popularity, acousticness)::numeric, 4)      AS acousticness,
        ROUND(CORR(target_popularity, instrumentalness)::numeric, 4)  AS instrumentalness,
        ROUND(CORR(target_popularity, liveness)::numeric, 4)          AS liveness,
        ROUND(CORR(target_popularity, valence)::numeric, 4)           AS valence,
        ROUND(CORR(target_popularity, tempo)::numeric, 4)             AS tempo
    FROM analytics.vw_ml_training_dataset
""", conn)

corr_series = df_corr.iloc[0].astype(float).sort_values(ascending=False)
print('=== Correlation với target_popularity ===')
for feat, val in corr_series.items():
    bar = '█' * int(abs(val) * 30)
    sign = '+' if val >= 0 else '-'
    print(f'  {feat:20s}: {sign}{abs(val):.4f}  {bar}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ca02c' if v >= 0 else '#d62728' for v in corr_series.values]
bars = ax.barh(corr_series.index, corr_series.values, color=colors)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Hệ số tương quan Pearson với target_popularity')
ax.set_title('Correlation của numeric features với target_popularity\n'
             '⚠️ target_popularity = LABEL — không dùng làm input', fontweight='bold')
for i, (feat, val) in enumerate(corr_series.items()):
    offset = 0.005 if val >= 0 else -0.005
    ax.text(val + offset, i, f'{val:.4f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=9)
ax.set_xlim(-0.55, 0.75)
plt.tight_layout()
plt.show()

## 2. Scatter Plot — Popularity vs Selected Features

Dùng aggregate bins để tránh plot 586K điểm.

In [ ]:
# Binned scatter: avg popularity theo bins của mỗi feature
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

feature_pairs = [
    ('release_year', 'Năm phát hành', False),
    ('loudness', 'Loudness (dB)', False),
    ('acousticness', 'Acousticness', True),
]

for ax, (feat, label, do_bin) in zip(axes, feature_pairs):
    if do_bin:
        sql = f"""
            SELECT ROUND(({feat})::numeric, 1) AS bin,
                   ROUND(AVG(target_popularity)::numeric, 2) AS avg_pop,
                   COUNT(*) AS cnt
            FROM analytics.vw_ml_training_dataset
            WHERE {feat} IS NOT NULL
            GROUP BY 1 ORDER BY 1
        """
    else:
        sql = f"""
            SELECT {feat} AS bin,
                   ROUND(AVG(target_popularity)::numeric, 2) AS avg_pop,
                   COUNT(*) AS cnt
            FROM analytics.vw_ml_training_dataset
            WHERE {feat} IS NOT NULL
            GROUP BY 1 ORDER BY 1
        """
    df_bin = pd.read_sql(sql, conn)
    ax.scatter(df_bin['bin'], df_bin['avg_pop'],
               s=df_bin['cnt'].apply(lambda x: min(float(x)/500, 30) + 2),
               alpha=0.6, color='steelblue')
    ax.set_xlabel(label)
    ax.set_ylabel('Avg popularity')
    ax.set_title(f'Popularity vs {label}', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Outlier Analysis

In [ ]:
df_outliers = pd.read_sql("""
    SELECT
        'Duration short (<10s)'                  AS outlier_type,
        COUNT(*) FILTER (WHERE duration_ms < 10000)          AS count,
        ROUND(COUNT(*) FILTER (WHERE duration_ms < 10000) * 100.0 / COUNT(*), 3) AS pct
    FROM analytics.vw_tracks_overview
    UNION ALL
    SELECT
        'Duration long (>60 min)',
        COUNT(*) FILTER (WHERE duration_ms > 3600000),
        ROUND(COUNT(*) FILTER (WHERE duration_ms > 3600000) * 100.0 / COUNT(*), 3)
    FROM analytics.vw_tracks_overview
    UNION ALL
    SELECT
        'Loudness > 0 dB',
        COUNT(*) FILTER (WHERE loudness > 0),
        ROUND(COUNT(*) FILTER (WHERE loudness > 0) * 100.0 / COUNT(*), 3)
    FROM analytics.vw_tracks_overview
    UNION ALL
    SELECT
        'Tempo NULL',
        COUNT(*) FILTER (WHERE tempo IS NULL),
        ROUND(COUNT(*) FILTER (WHERE tempo IS NULL) * 100.0 / COUNT(*), 3)
    FROM analytics.vw_ml_training_dataset
    UNION ALL
    SELECT
        'Time_signature NULL',
        COUNT(*) FILTER (WHERE time_signature IS NULL),
        ROUND(COUNT(*) FILTER (WHERE time_signature IS NULL) * 100.0 / COUNT(*), 3)
    FROM analytics.vw_ml_training_dataset
    UNION ALL
    SELECT
        'target_popularity = 0',
        COUNT(*) FILTER (WHERE target_popularity = 0),
        ROUND(COUNT(*) FILTER (WHERE target_popularity = 0) * 100.0 / COUNT(*), 2)
    FROM analytics.vw_ml_training_dataset
""", conn)

df_outliers['count'] = df_outliers['count'].apply(lambda x: f'{x:,}')
df_outliers['pct'] = df_outliers['pct'].apply(lambda x: f'{x:.3f}%')
df_outliers

In [ ]:
# Duration outlier summary chart
df_dur_all = pd.read_sql("""
    SELECT release_year, short_track_count, long_track_count
    FROM analytics.vw_duration_trends
    ORDER BY release_year
""", conn)

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(df_dur_all['release_year'], df_dur_all['long_track_count'],
       color='#d62728', alpha=0.8, label='Long (>60min)')
ax.bar(df_dur_all['release_year'], df_dur_all['short_track_count'],
       color='#ff7f0e', alpha=0.8, label='Short (<10s)')
ax.set_title('Duration Outliers theo năm\n⚠️ Carry-forward từ Feature 1.5: short=26, long=83',
             fontweight='bold')
ax.set_xlabel('Năm')
ax.set_ylabel('Số tracks')
ax.legend()
plt.tight_layout()
plt.show()
print('Tổng short:', df_dur_all['short_track_count'].sum())
print('Tổng long: ', df_dur_all['long_track_count'].sum())

## 4. Nhận xét & Kết luận

**Correlation với target_popularity:**

| Feature | Corr | Mức độ |
|---------|------|-------|
| `release_year` | **+0.591** | Mạnh — track mới được Spotify tính cao hơn |
| `loudness` | **+0.327** | Trung bình dương — track to hơn thường phổ biến hơn |
| `energy` | **+0.302** | Trung bình dương |
| `danceability` | **+0.187** | Yếu dương |
| `tempo` | **+0.072** | Rất yếu |
| `duration_min` | **+0.028** | Gần 0 |
| `valence` | **+0.005** | Gần 0 |
| `speechiness` | **−0.047** | Rất yếu âm |
| `liveness` | **−0.049** | Rất yếu âm |
| `instrumentalness` | **−0.237** | Yếu âm — nhạc instrumental ít popular hơn |
| `acousticness` | **−0.371** | Trung bình âm — nhạc acoustic có xu hướng older/less popular |

**Outliers — Rủi ro cho ML:**
- **Duration short (26) & long (83):** Ảnh hưởng nhỏ (<0.02%) — có thể filter ở EPIC 2.
- **Loudness > 0 (219):** Rất hiếm, có thể là liverecording hoặc metadata error — cân nhắc clip.
- **Tempo NULL (328) + Time_signature NULL (337):** Cần impute trước khi train.
- **target_popularity = 0 (44,690 / 7.6%):** Đây là inactive/unlisted tracks — không phải error. Quyết định giữ hoặc filter thuộc về ML strategy ở EPIC 2.

**Đề xuất cho Feature 1.8 / EPIC 2:**
1. `release_year` là feature dự báo mạnh nhất — cần kiểm tra data leakage.
2. Impute `tempo` (328) và `time_signature` (337) NULL.
3. Scale tất cả numeric features (MinMax hoặc StandardScaler).
4. Log-transform `speechiness`, `instrumentalness` do zero-inflation.
5. Quyết định giữ hay filter 44,690 tracks popularity=0.

In [ ]:
conn.close()
print('Done — Notebook 06 hoàn thành.')